In [1]:
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

from lib.main import *

from parameters import parameters

In [2]:
# Plate model name
plate_model_name = parameters["plate_model_name"]

# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

buffer_distance = parameters["buffer_distance"]
num_random = parameters["num_random"]
grid_resolution = parameters["grid_resolution"]

plate_model_dir = parameters["plate_model_dir"]
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]

subduction_data_filename = parameters["subduction_data_filename"]
deposit_coords_filename = parameters["deposit_coords_filename"]
deposit_recon_coords_filename = parameters["deposit_recon_coords_filename"]
deposit_recon_coords_all_filename = parameters["deposit_recon_coords_all_filename"]
unlabelled_coords_filename = parameters["unlabelled_coords_filename"]
target_coords_filename = parameters["target_coords_filename"]
deposit_data_filename = parameters["deposit_data_filename"]
unlabelled_data_filename = parameters["unlabelled_data_filename"]
target_data_filename = parameters["target_data_filename"]

subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)
deposit_coords_filename = os.path.join(inputs_dir, deposit_coords_filename)
deposit_recon_coords_filename = os.path.join(outputs_dir, deposit_recon_coords_filename)
deposit_recon_coords_all_filename = os.path.join(outputs_dir, deposit_recon_coords_all_filename)
buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)
unlabelled_coords_filename = os.path.join(outputs_dir, unlabelled_coords_filename)
target_coords_filename = os.path.join(outputs_dir, target_coords_filename)
deposit_data_filename = os.path.join(outputs_dir, deposit_data_filename)
unlabelled_data_filename = os.path.join(outputs_dir, unlabelled_data_filename)
target_data_filename = os.path.join(outputs_dir, target_data_filename)

agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
crusthick_dir = os.path.join(inputs_dir, "CrustalThickness")

nprocs = 16

In [3]:
subduction_data = pd.read_csv(subduction_data_filename)
deposit_coords = pd.read_csv(deposit_coords_filename)

plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

gplot = get_plot_topologies(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
    plate_reconstruction=plate_model,
    filter_topologies=True,
)

projection = ccrs.Mollweide(central_longitude=60)

### Buffer Zones

In [4]:
if not os.path.isdir(buffer_zones_dir):
    run_create_buffer_zones(
        nprocs=nprocs,
        times=time_steps,
        plate_reconstruction=plate_model,
        output_dir=buffer_zones_dir,
        buffer_distance=buffer_distance,
        verbose=True,
        return_output=False,
    )

In [5]:
@interact
def show_map(time=time_steps):
    gplot.time = time

    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5)),
    )

    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)

    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor='palegreen',
        edgecolor='none',
        alpha=0.7,
        zorder=4,
    )

    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color='k', zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    cb = fig.colorbar(im, orientation='horizontal', shrink=0.4, pad=0.06, extend='max')
    cb.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.25))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
    
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Reconstruct Mineral Occurrences

In [6]:
if os.path.isfile(deposit_recon_coords_filename):
    deposit_recon_coords = pd.read_csv(deposit_recon_coords_filename)
else:
    deposit_recon_coords = prepare_deposit_data(
        deposit_data=deposit_coords_filename,
        plate_reconstruction=plate_model,
        buffer_zones_dir=buffer_zones_dir,
        output_filename=deposit_recon_coords_filename,
        time_steps=time_steps,
        min_time=time_min,
        max_time=time_max,
        n_jobs=nprocs,
        verbose=True,
    )

In [7]:
if os.path.isfile(deposit_recon_coords_all_filename):
    deposit_recon_coords_all = pd.read_csv(deposit_recon_coords_all_filename)
else:
    deposit_recon_coords_all = partition_and_reconstruct(
        deposit_data=deposit_coords_filename,
        plate_reconstruction=plate_model,
        time_steps=time_steps,
        output_filename=deposit_recon_coords_all_filename,
        verbose=True,
    )

In [8]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove('lon')
features_plot.remove('lat')
features_plot.remove('age (Ma)')
features_plot.remove('subducting_plate_ID')
features_plot.remove('trench_plate_ID')

In [9]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    deposit_recon_coords_t = deposit_recon_coords_all[[f'lon_{time}', f'lat_{time}', 'weight']]
    deposit_recon_coords_t = deposit_recon_coords_t.dropna()
        
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    features_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5)),
    )

    cax_feat = fig.add_axes([0.1, 0.12, 0.35, 0.02])
    cax_bg = fig.add_axes([0.55, 0.12, 0.35, 0.02])
    
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    sc0 = ax.scatter(features_t['lon'], features_t['lat'], 50, marker='.',
                     c=features_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)
    
    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)

    sc1 = ax.scatter(
        deposit_recon_coords_t[f'lon_{time}'],
        deposit_recon_coords_t[f'lat_{time}'],
        transform=ccrs.PlateCarree(),
        marker='o',
        facecolor='yellow',
        edgecolor='black',
        s = [w * 10 for w in deposit_recon_coords_t['weight']],
        alpha=0.7,
        zorder=8
    )
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    cbar_feat = fig.colorbar(sc0, cax=cax_feat, orientation="horizontal")
    cbar_feat.set_label(feature, fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)

    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend='max')
    cbar_bg.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges'),
        Line2D([0], [0], marker='o', markerfacecolor='yellow', markeredgecolor='black', markersize=15, linestyle='None', label='Mineral Occurrence')
    ]

    # Add the custom legend to the plot
    legend = ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.15))
    
    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
    
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Unlabelled Samples

In [10]:
if os.path.isfile(unlabelled_coords_filename):
    unlabelled_coords = pd.read_csv(unlabelled_coords_filename)
else:
    unlabelled_coords = generate_unlabelled_points(
        times=time_steps,
        input_dir=buffer_zones_dir,
        num=num_random,
        threads=nprocs,
        seed=42,
        plate_reconstruction=plate_model,
        verbose=True,
    )
    
    unlabelled_coords = prepare_unlabelled_data(
        unlabelled_data=unlabelled_coords,
        plate_reconstruction=plate_model,
        output_filename=unlabelled_coords_filename,
        min_time=time_min,
        max_time=time_max,
        n_jobs=nprocs,
        verbose=True,
    )

In [11]:
@interact
def show_map(time=time_steps):
    gplot.time = time    
    
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('silver', alpha=0.5))
    )

    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)

    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor='palegreen',
        edgecolor='none',
        alpha=0.7,
        zorder=4,
    )
    
    unlabelled_coords_t = unlabelled_coords[unlabelled_coords["age (Ma)"] == time]

    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)

    ax.scatter(
        unlabelled_coords_t['lon'],
        unlabelled_coords_t['lat'],
        transform=ccrs.PlateCarree(),
        marker='X',
        facecolor='cyan',
        edgecolor='black',
        s=50,
        zorder=8
    )

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    cb = fig.colorbar(im, orientation='horizontal', shrink=0.4, pad=0.06, extend='max')
    cb.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)

    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.25))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Target Points

In [12]:
if os.path.isfile(target_coords_filename):
    target_coords = pd.read_csv(target_coords_filename)
    target_coords = target_coords.dropna(subset=["present_lon", "present_lat"])
else:
    target_coords = generate_grid_points(
        times=time_steps,
        resolution=grid_resolution,
        polygons_dir=buffer_zones_dir,
        plate_reconstruction=plate_model,
        output_filename=target_coords_filename,
        n_jobs=nprocs,
        verbose=True,
    )
    
    target_coords = target_coords.dropna(subset=["present_lon", "present_lat"])

In [13]:
@interact
def show_map(time=time_steps):
    gplot.time = time    
    
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5))
    )

    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)

    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    # buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    # buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    # buffer_zones_t.plot(
    #     ax=ax,
    #     transform=ccrs.PlateCarree(),
    #     facecolor='palegreen',
    #     edgecolor='none',
    #     alpha=0.7,
    #     zorder=4,
    # )

    target_coords_t = target_coords[target_coords["age (Ma)"] == time]

    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)

    ax.scatter(
        target_coords_t['lon'],
        target_coords_t['lat'],
        transform=ccrs.PlateCarree(),
        marker='.',
        c='red',
        s=1,
        alpha=0.5,
        zorder=6
    )
    
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=7)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=8)

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    cb = fig.colorbar(im, orientation='horizontal', shrink=0.4, pad=0.06, extend='max')
    cb.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        # Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges'),
        Line2D([0], [0], marker='.', markerfacecolor='red', markeredgecolor='none', markersize=10, linestyle='None', label='Target Points')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.25))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Coregistration

In [14]:
if not os.path.isfile(deposit_data_filename):
    deposit_data = run_coregister_point_data(
        point_data=deposit_recon_coords,
        subduction_data=subduction_data,
        n_jobs=nprocs,
        verbose=True,
    )
    
    deposit_data = run_coregister_crustal_thickness(
        point_data=deposit_data,
        input_dir=crusthick_dir,
        output_filename=deposit_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )

if not os.path.isfile(unlabelled_data_filename):
    unlabelled_data = run_coregister_point_data(
        point_data=unlabelled_coords,
        subduction_data=subduction_data,
        output_filename=unlabelled_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )
    
    unlabelled_data = run_coregister_crustal_thickness(
        point_data=unlabelled_data,
        input_dir=crusthick_dir,
        output_filename=unlabelled_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )

if not os.path.isfile(target_data_filename):
    target_data = run_coregister_point_data(
        point_data=target_coords,
        subduction_data=subduction_data,
        output_filename=target_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )
    
    target_data = run_coregister_crustal_thickness(
        point_data=target_data,
        input_dir=crusthick_dir,
        output_filename=target_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )